# Neural-network exercises

**Lab 5 · Notebook 3 of 3**

Complete **`Week_05_nns_in_pytorch.ipynb`** and **`Week_05_nns_with_torchnn.ipynb`** first. These four parts let you practise building and fitting the same neural network using the different implementations introduced there. Work in **Google Colab**. This notebook recreates the noisy-XOR dataset from the worked notebooks: four Gaussian clouds, with diagonally opposite clouds sharing a class label. Each observation receives the class label assigned to its generating cloud. No GPU is needed. Blue circles show class 0 and orange triangles show class 1.

Work from top to bottom, and try each question in its code cell before opening **Show answer**. The answer boxes are closed until you click their headings. To use an answer, copy its code into the corresponding code cell and run it. Opening the box does not execute the code. If you get stuck, use the supplied answer for that question and continue. Within each part, later questions use the results of the preceding ones.

Each part creates a **fresh 2 → 100 → 1 network** with one hidden layer and one output layer. The hidden layer applies an affine transformation followed by ReLU. The output layer produces an affine score, which we pass directly to the helper `loss_from_scores`. Use a learning rate of **`0.0001`** and **1,000 updates** throughout. The parts share the supplied data, but each builds its own model. To start a part again with fresh parameters, rerun its parameter or network construction and then work through that part in order.

Keep the notation from the worked notebooks: `W1`, `b1`, `W2`, and `b2` for explicit parameters, `a1` for hidden preactivations, `phi1` for hidden activations, and `scores` for the network output. We keep weights and biases separate, so `X` contains the two original features without a column of ones. The target tensor is `y`, with a lowercase name.

Recall that the log-loss in the notes takes a probability and a label. The helper `loss_from_scores(scores)` evaluates the whole objective

$$
\sum_{i=1}^{N}\ell_{\mathrm{log}}\bigl(\sigma(s_i),y_i\bigr),
$$

using the fixed labels in the tensor `y`. As explained in the first notebook, `logsigmoid` computes the required logarithms directly, without calculating probabilities as intermediate values. Pass the network's scores to `loss_from_scores`, without first applying a sigmoid. The mathematical loss is still the one defined in the notes.

Run the setup cell below. The data have shapes `(N, 2)` and `(N, 1)`. The plotting helper is supplied and remains **non-examinable**. You will call it after training each network, using the same data throughout.

In [ ]:
import torch as t
import torch.nn as nn
import matplotlib.pyplot as plt
t.manual_seed(1)  # Fix the random seed for this notebook.

N = 100
# Same noisy-XOR observations as the first notebook: 25 per Gaussian group.
# spread is the standard deviation in each input coordinate.
spread = 0.45
# Diagonally opposite groups share a class. Each label is the class assigned to the generating cloud.
x_class_0 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([-1.0, -1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0,  1.0])
], dim=0)
x_class_1 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([-1.0,  1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0, -1.0])
], dim=0)
X = t.cat([x_class_0, x_class_1], 0)              # Shape: (100, 2).
y = t.cat([t.zeros(N // 2, 1), t.ones(N // 2, 1)], 0)  # Shape: (100, 1).

# Show observed classes using the same colours and markers as in Lab 4.
plt.figure()
plt.scatter(x=x_class_0[:, 0], y=x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x=x_class_1[:, 0], y=x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.legend()
plt.show()

def loss_from_scores(scores):
    # scores contains the network's logits, not probabilities or weights.
    # Evaluate sum_i ell_log(sigmoid(score_i), y_i), using the fixed labels y.
    # logsigmoid computes the logarithms directly, without forming probabilities.
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

def plot_decision_boundary(net, X, y):
    # Non-examinable: display the fitted classifier and the observed class labels.
    # Use the same data-dependent limits for every network, including all observations.
    x1_min = X[:, 0].min().item() - 0.5
    x1_max = X[:, 0].max().item() + 0.5
    x2_min = X[:, 1].min().item() - 0.5
    x2_max = X[:, 1].max().item() + 0.5
    x1_grid, x2_grid = t.meshgrid(t.linspace(x1_min, x1_max, 1000),
                                t.linspace(x2_min, x2_max, 1000),
                                indexing="ij")
    X_grid = t.stack([x1_grid, x2_grid], dim=-1)

    # We only need predictions for this plot, not their gradients.
    with t.no_grad():
        c_grid = (net(X_grid) > 0)[..., 0]

    plt.figure()
    plt.contour(x1_grid, x2_grid, c_grid, levels=[0.5], colors='black', linewidths=0.7)
    class_0 = y[:, 0] == 0
    class_1 = y[:, 0] == 1
    plt.scatter(X[class_0, 0], X[class_0, 1], color='tab:blue', label='Class 0')
    plt.scatter(X[class_1, 0], X[class_1, 1], color='tab:orange', marker='^', label='Class 1')
    plt.xlim(x1_min, x1_max)
    plt.ylim(x2_min, x2_max)
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.legend()



## Part 1 — Explicit parameter tensors

### 1.1) Create the parameters

We begin by defining the parameter tensors directly, as in the first worked notebook. Create `W1`, `b1`, `W2`, and `b2` using `t.randn` with `requires_grad=True`. Use two input features, 100 hidden units, and one output score. For example, the first hidden layer needs one bias for each of its units.

Print the four parameter shapes and check them against the architecture. Then count the total number of adjustable scalar coefficients. The method `.numel()` returns the number of entries in a tensor, so adding the four counts gives the total. You should have **four parameter tensors containing 401 coefficients altogether**.

If you need a reminder, see Section 3.1 of **`Week_05_nns_in_pytorch.ipynb`**.

In [ ]:
# Your code for Question 1.1 goes here.

<details>
<summary>Show answer — 1.1</summary>

```python
input_features = 2
hidden_features = 100
output_features = 1

b1 = t.randn(hidden_features, requires_grad=True)
W1 = t.randn(input_features, hidden_features, requires_grad=True)
b2 = t.randn(output_features, requires_grad=True)
W2 = t.randn(hidden_features, output_features, requires_grad=True)

print("W1:", W1.shape)
print("b1:", b1.shape)
print("W2:", W2.shape)
print("b2:", b2.shape)
print("Number of parameters:", W1.numel() + b1.numel() + W2.numel() + b2.numel())
```

The shapes are `(2, 100)`, `(100,)`, `(100, 1)`, and `(1,)`, in the printed order. The biases are one-dimensional tensors, as in the worked notebooks. Broadcasting will add them to every observation's row.

The first layer has `2 * 100` weights and `100` biases. The output layer has `100 * 1` weights and one bias. Therefore, the total is `200 + 100 + 100 + 1 = 401`. Each entry is adjustable, so every one of them is a parameter.

</details>

### 1.2) Compute the forward pass

Write a function `net(X)` that uses the parameters from Question 1.1. First compute `a1`, then apply ReLU to obtain `phi1`, and finally combine those hidden activations to obtain `scores`.

The function should return the scores directly. Call it on the supplied `X` and print the result's shape. You should obtain `(N, 1)`, because each observation produces one score. The intermediate tensors `a1` and `phi1` should both have shape `(N, 100)`.

Create the parameters once in Question 1.1, and reuse them each time the function is called.

In [ ]:
# Your code for Question 1.2 goes here.

<details>
<summary>Show answer — 1.2</summary>

```python
def net(X):
    a1 = X @ W1 + b1
    phi1 = t.relu(a1)
    scores = phi1 @ W2 + b2
    return scores

scores = net(X)
print(scores.shape)
```

The output is `torch.Size([100, 1])`. The first affine transformation produces 100 values for each of the 100 observations. ReLU changes their values while preserving that shape. The final affine transformation combines each row into one score.

The function refers to the four existing parameter tensors. Therefore, subsequent calls use their current values, including any changes made during training.

</details>

### 1.3) Compute the loss and its gradients

Recompute `scores = net(X)` each time you run this check, then evaluate the supplied `loss_from_scores(scores)`. This provides a fresh loss for `L.backward()`; do not reuse scores whose loss has already been used in an earlier call to `backward()`. Store the result as `L`, print its value and shape, and call `L.backward()` to compute the gradients. Then print the shapes of `W1.grad`, `b1.grad`, `W2.grad`, and `b2.grad`.

The loss should be a finite scalar tensor, whose shape prints as `torch.Size([])`. Each gradient should have the same shape as its parameter, because it contains one derivative per parameter entry.

Remember that `backward()` adds to any stored gradients. Before this calculation, clear each gradient that already exists. A fresh parameter has `.grad` equal to `None`, so use `if parameter.grad is not None:` before calling `parameter.grad.zero_()`. This also allows you to rerun your check without accumulating gradients from earlier attempts.

In [ ]:
# Your code for Question 1.3 goes here.

<details>
<summary>Show answer — 1.3</summary>

```python
for parameter in [W1, b1, W2, b2]:
    if parameter.grad is not None:
        parameter.grad.zero_()

scores = net(X)
L = loss_from_scores(scores)
print("Loss:", L.item())
print("Loss shape:", L.shape)
L.backward()

print("W1 gradient:", W1.grad.shape)
print("b1 gradient:", b1.grad.shape)
print("W2 gradient:", W2.grad.shape)
print("b2 gradient:", b2.grad.shape)
```

The loss has shape `torch.Size([])`. Its numerical value depends on the current parameters, but it should be finite for the supplied setup. The gradient shapes are `(2, 100)`, `(100,)`, `(100, 1)`, and `(1,)`.

Calling `.item()` only provides the number that we print. The variable `L` remains a tensor, so `L.backward()` can still compute its gradients. This call does not change the parameters. We will clear these stored gradients before starting the training loop.

</details>

### 1.4) Train the network

Use gradient descent with learning rate `0.0001` for 1,000 updates. Clear the gradients left by Question 1.3 before entering the loop. In each iteration, compute a fresh loss, call `backward()`, update all four parameter tensors inside `with t.no_grad():`, and then clear their gradients.

Record the loss before training as `initial_loss`. Print the current loss after updates 100, 200, and so on through 1,000. After the last update, recompute the loss as `final_loss` and print the initial and final values together. Recomputing matters because the tensor used for `backward()` describes the parameters before that iteration's update.

With the supplied setup, the final loss should be smaller than the initial loss. If it is not, check the sign of the update, whether all four tensors are updated, and whether old gradients are being cleared. A lower loss does not require the final loss to be zero.

In [ ]:
# Your code for Question 1.4 goes here.

<details>
<summary>Show answer — 1.4</summary>

```python
# Clear the gradients computed in Question 1.3.
W1.grad.zero_()
b1.grad.zero_()
W2.grad.zero_()
b2.grad.zero_()

learning_rate = 0.0001
initial_loss = loss_from_scores(net(X)).item()

for step in range(1, 1001):
    L = loss_from_scores(net(X))
    L.backward()

    with t.no_grad():
        W1.add_(-learning_rate * W1.grad)
        b1.add_(-learning_rate * b1.grad)
        W2.add_(-learning_rate * W2.grad)
        b2.add_(-learning_rate * b2.grad)

    W1.grad.zero_()
    b1.grad.zero_()
    W2.grad.zero_()
    b2.grad.zero_()

    if step % 100 == 0:
        print("After", step, "updates:", loss_from_scores(net(X)).item())

final_loss = loss_from_scores(net(X)).item()
print("Initial loss:", initial_loss)
print("Final loss:", final_loss)
```

Each call to `L.backward()` computes all four gradients before any parameter is updated. Consequently, the four updates use derivatives evaluated at the same current parameter values. The resets then prepare the stored gradients for the next iteration.

The printed losses are evaluated after the reported number of updates. The final loss is evaluated after update 1,000, so it measures the network that you will inspect in the next question. If you run this training cell again, it continues from the current parameters. To make a fresh start, repeat Questions 1.1–1.3 first.

</details>

### 1.5) Inspect the predictions and the decision boundary

First recompute `scores = net(X)` using the trained parameters, then compute the predicted labels for these observations. The scores saved before training do not update automatically when the parameters change. As in the notes and Lab 4, predict class 1 when a score is positive, and class 0 otherwise, including a zero score. The comparison `scores > 0` produces Boolean values. Calling `.to(y.dtype)` expresses those labels as zeros and ones with the same data type as `y`.

Count the observations for which the predicted label differs from `y`. Both tensors should have shape `(N, 1)`. You can use `(predicted_labels != y).sum().item()` to count the mistakes. Then call `plot_decision_boundary(net, X, y)` and finish with `plt.show()`.

Describe how the boundary separates the four clouds. Why can a single straight boundary not separate the four cloud centres according to their labels? Also explain why the logistic loss and the number of classification mistakes are different quantities. These checks use the training data, so they describe how the fitted network behaves on the observations it has seen.

In [ ]:
# Your code for Question 1.5 goes here.

<details>
<summary>Show answer — 1.5</summary>

```python
with t.no_grad():
    scores = net(X)
    predicted_labels = (scores > 0).to(y.dtype)
    mistakes = (predicted_labels != y).sum().item()
    final_loss = loss_from_scores(scores).item()

print("Final loss:", final_loss)
print("Training mistakes:", mistakes, "out of", N)
plot_decision_boundary(net, X, y)
plt.show()
```

The mistake count is an integer between 0 and 100. Its value, and the precise boundary, depend on the fitted parameters. The helper draws a grid approximation of the boundary where the predicted class changes. Blue circles show observations labelled 0, and orange triangles show observations labelled 1, as in Lab 4. To connect an individual mistake with the figure, compare that observation's predicted label with its observed label.

Diagonally opposite clouds share a label, while neighbouring clouds have different labels. A single straight boundary cannot separate the four centres according to these labels. The network can form a boundary from several straight segments. Its precise shape and the number of mistakes depend on the fitted parameters.

The mistake count depends only on whether each predicted class is correct. The logistic loss also depends on the probabilities assigned to the observed classes. Thus the loss can change even when none of the predicted labels changes, and a correct classification can still contribute a positive loss.

</details>

## Part 2 — Your own module classes

We now organise the same calculations using `MyLinear`, `MyReLU`, and `MySequential`. Follow Sections 1–3 of **`Week_05_nns_with_torchnn.ipynb`** if you need a reminder.

Run the supplied `MySequential` definition below. It is the same implementation as in the worked notebook. You will define the two kinds of module that it combines, then build a fresh network and train it. The first affine module followed by ReLU forms hidden layer 1, while the final affine module forms output layer 2.

In [ ]:
class MySequential:
    def __init__(self, modules):
        """Store a list of modules, such as linear layers and ReLUs."""
        self.modules = modules

    def __call__(self, x):
        """Apply each module in order. Two modules give modules[1](modules[0](x))."""
        for mod in self.modules:
            x = mod(x)  # Feed this module's output to the next module.
        return x

    def parameters(self):
        """Collect the parameter tensors from every module."""
        full_list = []
        for mod in self.modules:
            for param in mod.parameters():
                full_list.append(param)  # Add this tensor to the list.
        return full_list

### 2.1) Define and check the modules

Define `MyLinear` with `__init__`, `__call__`, and `parameters()` methods. Its weights should be stored as `self.W` with shape `(in_features, out_features)`, and its biases as `self.b` with shape `(out_features,)`. Create both tensors with `requires_grad=True`. Calling the object should apply its affine transformation, and `parameters()` should return its two existing parameter tensors.

Then define `MyReLU`. Calling it should apply ReLU, and its `parameters()` method should return an empty list because it has no adjustable coefficients.

Before building the full network, test the classes with an affine module taking two inputs to three outputs, followed by a ReLU module. Applying each to the appropriate values from `X` should produce shape `(N, 3)`. Check that the affine module returns two parameter tensors and ReLU returns none.

In [ ]:
# Your code for Question 2.1 goes here.

<details>
<summary>Show answer — 2.1</summary>

```python
class MyLinear:
    def __init__(self, in_features, out_features):
        self.W = t.randn(in_features, out_features, requires_grad=True)
        self.b = t.randn(out_features, requires_grad=True)

    def __call__(self, x):
        return x @ self.W + self.b

    def parameters(self):
        return [self.W, self.b]

class MyReLU:
    def __call__(self, x):
        return t.relu(x)

    def parameters(self):
        return []

test_linear = MyLinear(2, 3)
test_relu = MyReLU()
a_test = test_linear(X)
phi_test = test_relu(a_test)
print("Affine output:", a_test.shape)
print("ReLU output:", phi_test.shape)
print("Affine parameter tensors:", len(test_linear.parameters()))
print("ReLU parameters:", test_relu.parameters())
```

The two shapes are both `torch.Size([100, 3])`. The last two outputs are `2` and `[]`.

The parameter tensors are created in `__init__` and reused by `__call__`. Returning these same tensors from `parameters()` allows the training loop to update the coefficients used by subsequent calls. The small test uses its own module instances. We will create new instances for the full network.

</details>

### 2.2) Build and check the network

Use `MySequential` to construct a fresh network named `net` with the same 2 → 100 → 1 architecture. Its list should contain an affine module, a ReLU module, and another affine module.

Check the shape of `net(X)`, print the shapes returned by `net.parameters()`, and count the total number of parameter entries. You should find `(N, 1)` scores and four parameter tensors containing 401 coefficients altogether. The list-comprehension pattern `[p.shape for p in net.parameters()]` from the worked notebook lets you inspect the four shapes.

In [ ]:
# Your code for Question 2.2 goes here.

<details>
<summary>Show answer — 2.2</summary>

```python
net = MySequential([
    MyLinear(2, 100),
    MyReLU(),
    MyLinear(100, 1)
])

print("Score shape:", net(X).shape)
print("Parameter shapes:", [p.shape for p in net.parameters()])
parameter_count = 0
for parameter in net.parameters():
    parameter_count += parameter.numel()
print("Number of parameters:", parameter_count)
```

The score shape is `torch.Size([100, 1])`. The parameter shapes are `(2, 100)`, `(100,)`, `(100, 1)`, and `(1,)`, and the total count is 401.

These are the same shapes as in Part 1. The first `MyLinear` stores the tensors playing the roles of `W1` and `b1`, and the second stores those playing the roles of `W2` and `b2`. `MySequential.parameters()` collects them, while ReLU contributes an empty list.

</details>

### 2.3) Train and inspect the new network

Adapt the training loop from Question 1.4 so that it updates the tensors returned by `net.parameters()`. Use a loop over those tensors for the updates and for clearing gradients. Before training, use the `if parameter.grad is not None:` check from Question 1.3 when clearing gradients: no gradients have been computed for the new network yet, so its `.grad` attributes initially equal `None`. The learning rate, number of updates, and loss remain the same.

Print the initial and final losses, with progress after every 100 updates. Then reuse the calculations from Question 1.5 to count training mistakes and draw the decision boundary. The final loss should be lower than the initial loss with the supplied setup.

The new network has fresh random parameters. Therefore, its numerical losses and mistake count need not match Part 1, even though the architecture and learning problem are the same.

In [ ]:
# Your code for Question 2.3 goes here.

<details>
<summary>Show answer — 2.3</summary>

```python
# Clear any gradients retained from an earlier attempt.
for parameter in net.parameters():
    if parameter.grad is not None:
        parameter.grad.zero_()

learning_rate = 0.0001
initial_loss = loss_from_scores(net(X)).item()

for step in range(1, 1001):
    L = loss_from_scores(net(X))
    L.backward()

    with t.no_grad():
        for parameter in net.parameters():
            parameter.add_(-learning_rate * parameter.grad)

    for parameter in net.parameters():
        parameter.grad.zero_()

    if step % 100 == 0:
        print("After", step, "updates:", loss_from_scores(net(X)).item())

final_loss = loss_from_scores(net(X)).item()
print("Initial loss:", initial_loss)
print("Final loss:", final_loss)

with t.no_grad():
    scores = net(X)
    predicted_labels = (scores > 0).to(y.dtype)
    mistakes = (predicted_labels != y).sum().item()

print("Training mistakes:", mistakes, "out of", N)
plot_decision_boundary(net, X, y)
plt.show()
```

The loop over `net.parameters()` updates the same four kinds of coefficient that we previously named individually. Each call to `L.backward()` still computes their gradients before the updates, and each reset prepares them for the next iteration.

The final checks have the same meanings as in Part 1. A lower logistic loss indicates a better value of the training objective. The mistake count records class predictions on the training data, while the graph shows the boundary produced by this particular fitted network.

</details>

## Part 3 — The built-in `torch.nn` classes

### 3.1) Construct the network with built-in modules

Use `nn.Sequential`, `nn.Linear`, and `nn.ReLU` to create a fresh network named `net` with the same 2 → 100 → 1 architecture. The setup cell has already imported `torch.nn as nn`.

Check the output shape, the four parameter shapes, and the total parameter count, as in Question 2.2. Remember that `nn.Sequential` takes its modules as separate arguments. Also remember that a built-in `nn.Linear` stores its weights with shape `(out_features, in_features)`, while our `MyLinear` used the opposite orientation.

The output should still have shape `(N, 1)`, and the network should still contain 401 parameters. Section 4 of **`Week_05_nns_with_torchnn.ipynb`** contains the corresponding worked example.

In [ ]:
# Your code for Question 3.1 goes here.

<details>
<summary>Show answer — 3.1</summary>

```python
net = nn.Sequential(
    nn.Linear(2, 100),
    nn.ReLU(),
    nn.Linear(100, 1)
)

print("Score shape:", net(X).shape)
print("Parameter shapes:", [p.shape for p in net.parameters()])
parameter_count = 0
for parameter in net.parameters():
    parameter_count += parameter.numel()
print("Number of parameters:", parameter_count)
```

The score shape is `torch.Size([100, 1])`. The parameter shapes are now `(100, 2)`, `(100,)`, `(1, 100)`, and `(1,)`. The total remains 401.

The changed weight orientation follows the storage convention of `nn.Linear`. It does not change the number of coefficients or the input and output dimensions. The built-in modules use the attribute names `.weight` and `.bias`.

</details>

### 3.2) Reuse the training and inspection calculations

Train the new network using the loop from Question 2.3, then compare its initial and final losses, count its training mistakes, and plot its decision boundary.

The training code can still iterate over `net.parameters()`, even though those tensors are now stored by PyTorch's built-in modules. Use the same loss, learning rate, and 1,000 updates. Consider why the same loop works for both implementations.

As explained in the worked notebook, `nn.Linear` uses a different random initialisation from our custom class. Compare the initial and final loss of this network with each other. You should not expect numerical equality with the preceding parts.

In [ ]:
# Your code for Question 3.2 goes here.

<details>
<summary>Show answer — 3.2</summary>

```python
# Clear any gradients retained from an earlier attempt.
for parameter in net.parameters():
    if parameter.grad is not None:
        parameter.grad.zero_()

learning_rate = 0.0001
initial_loss = loss_from_scores(net(X)).item()

for step in range(1, 1001):
    L = loss_from_scores(net(X))
    L.backward()

    with t.no_grad():
        for parameter in net.parameters():
            parameter.add_(-learning_rate * parameter.grad)

    for parameter in net.parameters():
        parameter.grad.zero_()

    if step % 100 == 0:
        print("After", step, "updates:", loss_from_scores(net(X)).item())

final_loss = loss_from_scores(net(X)).item()
print("Initial loss:", initial_loss)
print("Final loss:", final_loss)

with t.no_grad():
    scores = net(X)
    predicted_labels = (scores > 0).to(y.dtype)
    mistakes = (predicted_labels != y).sum().item()

print("Training mistakes:", mistakes, "out of", N)
plot_decision_boundary(net, X, y)
plt.show()
```

The loop works because both implementations expose their adjustable tensors through `parameters()`, and both compute their scores using PyTorch operations that support automatic differentiation.

The model construction has changed, but we still evaluate the same logistic loss, calculate gradients, and subtract the learning rate times each gradient. As before, inspect the final loss, mistake count, and boundary together. They describe related aspects of the fitted classifier, rather than three interchangeable measures.

</details>

## Part 4 — Your own `nn.Module` network

### 4.1) Define and check a network class

Define `MyNet` as a subclass of `nn.Module`, using built-in `nn.Linear` and `nn.ReLU` objects. Copy the subclassing template from Sections 5–6 of **`Week_05_nns_with_torchnn.ipynb`**. The inheritance machinery remains non-examinable, as in Lab 2. Here the task is to choose the layers and write the network calculation using that template.

In `__init__`, call `super().__init__()` and create `self.layer1`, `self.relu`, and `self.layer2`. Let the constructor arguments specify the input, hidden, and output sizes. In `forward`, compute `a1` with the first affine module, `phi1` with ReLU, and `scores` with the final affine module. Return the scores.

Create a fresh 2 → 100 → 1 instance named `net`. Check the output and parameter shapes and the total parameter count. Temporarily print `phi1.shape` inside `forward` and call `net(X)` once to inspect the hidden activations. Then comment out that print before training or plotting, since `forward` will be called repeatedly.

In [ ]:
# Your code for Question 4.1 goes here.

<details>
<summary>Show answer — 4.1</summary>

```python
class MyNet(nn.Module):
    def __init__(self, input_features, hidden_features, output_features):
        super().__init__()
        self.layer1 = nn.Linear(input_features, hidden_features)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_features, output_features)

    def forward(self, x):
        a1 = self.layer1(x)
        phi1 = self.relu(a1)
        # print(phi1.shape)  # Uncomment to inspect one call, then comment out again.
        scores = self.layer2(phi1)
        return scores

net = MyNet(2, 100, 1)
print("Score shape:", net(X).shape)
print("Parameter shapes:", [p.shape for p in net.parameters()])
parameter_count = 0
for parameter in net.parameters():
    parameter_count += parameter.numel()
print("Number of parameters:", parameter_count)
```

The score and parameter shapes are the same as in Part 3, and the parameter count is again 401. When you temporarily uncomment the print inside `forward`, passing the dataset `X` prints `torch.Size([100, 100])`: one row per observation and one column per hidden unit. Leave that line commented out for the remaining calculations.

Assigning the two affine modules to the network registers them as submodules, so the inherited `net.parameters()` method finds their weights and biases. Calling `net(X)` runs `forward` through the normal `nn.Module` calling mechanism. The three calculations are the same as before, but are now written explicitly in the network class.

</details>

### 4.2) Train and inspect the network class

Train this final network with the same loss, learning rate, and 1,000 updates. As in Section 6 of the worked notebook, you can use `net.zero_grad()` to clear the stored gradients for the whole model. Call it before training and after each update. It clears the gradients while keeping the parameter values.

Print the initial and final losses and the progress after every 100 updates. Finally, count the training mistakes and plot the decision boundary using the same inspection code as before.

Compare the role of `forward` here with the sequence of modules in Part 3. Which calculation did you have to write explicitly, and which parts of the training procedure stayed the same?

In [ ]:
# Your code for Question 4.2 goes here.

<details>
<summary>Show answer — 4.2</summary>

```python
net.zero_grad()
learning_rate = 0.0001
initial_loss = loss_from_scores(net(X)).item()

for step in range(1, 1001):
    L = loss_from_scores(net(X))
    L.backward()

    with t.no_grad():
        for parameter in net.parameters():
            parameter.add_(-learning_rate * parameter.grad)

    net.zero_grad()

    if step % 100 == 0:
        print("After", step, "updates:", loss_from_scores(net(X)).item())

final_loss = loss_from_scores(net(X)).item()
print("Initial loss:", initial_loss)
print("Final loss:", final_loss)

with t.no_grad():
    scores = net(X)
    predicted_labels = (scores > 0).to(y.dtype)
    mistakes = (predicted_labels != y).sum().item()

print("Training mistakes:", mistakes, "out of", N)
plot_decision_boundary(net, X, y)
plt.show()
```

The method `forward` specifies how the stored modules are applied: first the affine transformation, then ReLU, and finally the output transformation. In Part 3, `nn.Sequential` supplied this sequence for us.

The loss, gradient calculation, gradient-descent update, and prediction rule remain the same. `net.zero_grad()` clears the gradients for all registered parameters at once. As in the worked notebook, this can set their `.grad` attributes to `None`, and the next call to `backward()` computes and stores new gradients there.

All four parts represent the same family of 2 → 100 → 1 networks. They organise the computation differently, while different random initial parameters can lead to different fitted losses and boundaries.

</details>